# Big Query SQLX - Subscription Analytics Pipeline

This project uses **Dataform in BigQuery** to build a structured subscription analytics workflow.  
The repository is organized inside a **workspace**, and all SQLX models are placed under the `definitions` directory in clear layered folders.

## Project Structure

The project follows a layered design so that raw data is first ingested, then cleaned, transformed, enriched, and finally published as reporting-ready marts.

## Folder Structure

```text
definitions/
├── raw/
│   └── customer_cases.sqlx
├── staging/
│   └── ...
├── core/
│   └── ...
├── enriched/
│   └── ...
└── marts/
    └── ...

#### definitions/raw/customer_cases.sqlx

In [ ]:
config {
    type: "declaration",
    database: "lucid-arch-464008-n1",
    schema: "raw_layer",
    name: "customer_cases"
}

#### definitions/raw/customer_info.sqlx

In [ ]:
config {
    type: "declaration",
    database: "lucid-arch-464008-n1",
    schema: "raw_layer",
    name: "customer_info"
}

#### definitions/raw/customer_product.sqlx

In [ ]:
config {
    type: "declaration",
    database: "lucid-arch-464008-n1",
    schema: "raw_layer",
    name: "customer_product"
}

#### definitions/raw/product_info.sqlx

In [ ]:
config {
    type: "declaration",
    database: "lucid-arch-464008-n1",
    schema: "raw_layer",
    name: "product_info"
}

#### definitions/staging/stg_customer_cases.sqlx

In [ ]:
config {
    type: "table",
    schema: "staging_layer"
}

SELECT
  case_id,
  date_time,
  customer_id,
  LOWER(TRIM(channel)) AS channel,
  LOWER(TRIM(reason)) AS reason
FROM
  ${ref("customer_cases")}
WHERE
  case_id IS NOT NULL

#### definitions/staging/stg_customer_info.sqlx


In [ ]:
config {
    type: "table",
    schema: "staging_layer"
}

SELECT
  customer_id,
  age,
  LOWER(TRIM(gender)) AS gender
FROM
  ${ref("customer_info")}
WHERE
  customer_id IS NOT NULL

#### definitions/staging/stg_customer_product.sqlx

In [ ]:
config {
    type: "table",
    schema: "staging_layer"
}

SELECT
  customer_id,
  product,
  signup_date_time,
  cancel_date_time
FROM
  ${ref("customer_product")}
WHERE
  customer_id IS NOT NULL

#### definitions/staging/stg_product_info.sqlx

In [ ]:
config {
    type: "table",
    schema: "staging_layer"
}

SELECT
  product_id,
  name AS product_name,
  price,
  billing_cycle
FROM
  ${ref("product_info")}
WHERE
  product_id IS NOT NULL

#### definitions/core/dim_customer.sqlx

In [ ]:
config {
    type: "table",
    schema: "core_layer"
}

SELECT
  DISTINCT customer_id,
  age,
  gender,
  CASE
    WHEN gender = 'male' THEN 1
    ELSE 0
END
  AS is_male,
  CASE
    WHEN age > 60 THEN 'senior_citizen'
    ELSE 'working_citizen'
END
  AS subscriber_age_type,
  CASE
    WHEN age > 60 THEN 1
    ELSE 0
END
  AS is_senior_citizen
FROM
  ${ref("stg_customer_info")}
WHERE
  customer_id IS NOT NULL

#### definitions/core/dim_product.sqlx


In [ ]:
config {
    type: "table",
    schema: "core_layer"
}

SELECT
  DISTINCT product_id,
  product_name,
  price,
  billing_cycle,
  CASE
    WHEN billing_cycle = 12 THEN 1
    ELSE 0
END
  is_annually,
  CASE
    WHEN billing_cycle = 1 THEN 1
    ELSE 0
END
  is_monthly,
  price / NULLIF(billing_cycle, 0) AS price_per_month,
  CASE
    WHEN product_id = 'prd_1' AND product_name = 'annual_subscription' THEN 'prd_1_is_annual'
    ELSE CONCAT(product_id, '_is_', REPLACE(product_name, '_subscription', ''))
END
  AS product_subscription_type
FROM
  ${ref("stg_product_info")}
WHERE
  product_id IS NOT NULL

#### definitions/core/fct_customer_case.sqlx

In [ ]:
config {
    type: "table",
    schema: "core_layer"
}

SELECT
  case_id,
  customer_id,
  SAFE_CAST(date_time AS DATE) AS case_date,
  channel,
  reason,
  CASE
    WHEN channel = 'phone' THEN 1
    ELSE 0
END
  AS is_phone,
  CASE
    WHEN reason = 'support' THEN 1
    ELSE 0
END
  AS is_support,
  CASE
    WHEN channel = 'phone' AND reason = 'support' THEN 1
    ELSE 0
END
  AS is_phone_support_case,
  CASE
    WHEN channel = 'email' AND reason = 'support' THEN 1
    ELSE 0
END
  AS is_email_support_case,
  CASE
    WHEN channel = 'phone' AND reason = 'signup' THEN 1
    ELSE 0
END
  AS is_phone_signup_case,
  CASE
    WHEN channel = 'email' AND reason = 'signup' THEN 1
    ELSE 0
END
  AS is_email_signup_case
FROM
  ${ref("stg_customer_cases")}
WHERE
  case_id IS NOT NULL

#### definitions/core/fct_subscription.sqlx

In [ ]:
config {
    type: "table",
    schema: "core_layer"
}

SELECT
  customer_id,
  product,
  signup_date_time AS signup_timestamp,
  CASE
    WHEN cancel_date_time = 'NA' THEN NULL
    ELSE SAFE_CAST(cancel_date_time AS TIMESTAMP)
END
  AS cancel_timestamp,
  CASE
    WHEN cancel_date_time = 'NA' OR cancel_date_time IS NULL THEN 1
    ELSE 0
END
  AS is_active,
  SAFE_CAST(signup_date_time AS DATE) AS signup_date,
  CASE
    WHEN cancel_date_time = 'NA' THEN NULL
    ELSE SAFE_CAST(SAFE_CAST(cancel_date_time AS TIMESTAMP) AS DATE)
END
  AS cancel_date,
  CASE
    WHEN cancel_date_time = 'NA' THEN DATE_DIFF(CURRENT_DATE(), SAFE_CAST(signup_date_time AS DATE), DAY)
    ELSE DATE_DIFF(SAFE_CAST(SAFE_CAST(cancel_date_time AS TIMESTAMP) AS DATE), SAFE_CAST(signup_date_time AS DATE), DAY)
END
  AS active_days
FROM
  ${ref("stg_customer_product")}
WHERE
  customer_id IS NOT NULL

#### definitions/enriched/enriched_customer_case.sqlx

In [ ]:
config {
    type: "table",
    schema: "enrichment_layer"
}

SELECT
  f.case_id,
  f.customer_id,
  f.case_date,
  f.channel,
  f.reason,
  f.is_phone,
  f.is_support,
  f.is_phone_support_case,
  f.is_email_support_case,
  f.is_phone_signup_case,
  f.is_email_signup_case,
  c.age,
  c.gender,
  c.is_male,
  c.subscriber_age_type,
  c.is_senior_citizen
FROM
  ${ref("fct_customer_case")} AS f
LEFT JOIN
  ${ref("dim_customer")} AS c
ON
  f.customer_id = c.customer_id

#### definitions/enriched/enriched_subscription.sqlx

In [ ]:
config {
    type: "table",
    schema: "enrichment_layer"
}

SELECT
  s.customer_id,
  s.product AS product_id,
  s.signup_timestamp,
  s.cancel_timestamp,
  s.is_active,
  s.signup_date,
  s.cancel_date,
  s.active_days,
  c.age,
  c.gender,
  c.is_male,
  c.subscriber_age_type,
  c.is_senior_citizen,
  p.product_name,
  p.price,
  p.billing_cycle,
  p.is_annually,
  p.is_monthly,
  p.price_per_month,
  p.product_subscription_type
FROM
  ${ref("fct_subscription")} AS s
LEFT JOIN
  ${ref("dim_customer")} AS c
ON
  s.customer_id = c.customer_id
LEFT JOIN
  ${ref("dim_product")} AS p
ON
  s.product = p.product_id

#### definitions/marts/customer_360.sqlx


In [ ]:
config {
    type: "view",
    schema: "mart_layer",
}

WITH
  sub_agg AS (
  SELECT
    customer_id,
    COUNT(DISTINCT product_id) AS total_products,
    SUM(is_active) AS active_subscriptions,
    AVG(active_days) AS avg_active_days,
    MAX(is_senior_citizen) AS is_senior_citizen
  FROM
    ${ref("enriched_subscription")}
  GROUP BY
    1 ),
  case_agg AS (
  SELECT
    customer_id,
    COUNT(case_id) AS total_cases,
    SUM(is_support) AS total_support_cases,
    SUM(is_phone_support_case) AS phone_support_cases
  FROM
    ${ref("enriched_customer_case")}
  GROUP BY
    1 )
SELECT
  COALESCE(s.customer_id, c.customer_id) AS customer_id,
  s.total_products,
  s.active_subscriptions,
  s.avg_active_days,
  s.is_senior_citizen,
  COALESCE(c.total_cases, 0) AS total_cases,
  COALESCE(c.total_support_cases, 0) AS total_support_cases,
  COALESCE(c.phone_support_cases, 0) AS phone_support_cases
FROM
  sub_agg s
FULL OUTER JOIN
  case_agg c
ON
  s.customer_id = c.customer_id

#### definitions/marts/monthly_executive_kpi.sqlx


In [ ]:
config {
    type: "view",
    schema: "mart_layer",
}

WITH
  signups AS (
  SELECT
    DATE_TRUNC(signup_date, MONTH) AS kpi_month,
    COUNT(*) AS new_subs
  FROM
    ${ref("enriched_subscription")}
  GROUP BY
    1 ),
  cancels AS (
  SELECT
    DATE_TRUNC(cancel_date, MONTH) AS kpi_month,
    COUNT(*) AS cancelled_subs
  FROM
    ${ref("enriched_subscription")}
  WHERE
    cancel_date IS NOT NULL
  GROUP BY
    1 ),
  cases AS (
  SELECT
    DATE_TRUNC(case_date, MONTH) AS kpi_month,
    COUNT(*) AS total_cases
  FROM
    ${ref("enriched_customer_case")}
  GROUP BY
    1 )
SELECT
  COALESCE(s.kpi_month, c.kpi_month, cs.kpi_month) AS kpi_month,
  COALESCE(s.new_subs, 0) AS new_subscriptions,
  COALESCE(c.cancelled_subs, 0) AS cancelled_subscriptions,
  COALESCE(cs.total_cases, 0) AS total_cases
FROM
  signups s
FULL OUTER JOIN
  cancels c
ON
  s.kpi_month = c.kpi_month
FULL OUTER JOIN
  cases cs
ON
  COALESCE(s.kpi_month, c.kpi_month) = cs.kpi_month
WHERE
  COALESCE(s.kpi_month, c.kpi_month, cs.kpi_month) IS NOT NULL

#### definitions/marts/product_performance.sqlx


In [ ]:
config {
    type: "view",
    schema: "mart_layer",
}

SELECT
  product_id,
  product_name,
  product_subscription_type,
  price_per_month,
  COUNT(customer_id) AS total_subscriptions,
  SUM(is_active) AS active_count,
  COUNTIF(is_active = 0) AS churned_count,
  ROUND(SAFE_DIVIDE(COUNTIF(is_active = 0), COUNT(*)) * 100, 2) AS churn_rate_pct
FROM
  ${ref("enriched_subscription")}
GROUP BY
  1,
  2,
  3,
  4

#### definitions/marts/support_metrics.sqlx


In [ ]:
config {
    type: "view",
    schema: "mart_layer",
}

SELECT
  case_date,
  channel,
  reason,
  subscriber_age_type,
  COUNT(case_id) AS total_cases,
  SUM(is_phone_support_case) AS phone_support_cases,
  SUM(is_email_support_case) AS email_support_cases
FROM
  ${ref("enriched_customer_case")}
GROUP BY
  1,
  2,
  3,
  4